# Post-descriptor processing (variance, correlation and scaling)

In [ ]:
import pandas as pd         # imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.under_sampling import RepeatedEditedNearestNeighbours as RENN, NearMiss

In [ ]:
main = pd.read_csv(r'M:\ML_scripts\mordred_mold2_rdkit_descr_tob_5889.csv').drop(columns=['Unnamed: 0'])  # clean data with Mordred/Mold2/RDKit descriptors
main_y = main['Class']

full_train, full_test = train_test_split(main, test_size=0.2, stratify=main_y, random_state=42) # split data into train and test sets, stratified split based on the target variable (class)

# train set processing

df_str = full_train['Structure'] # smiles
df_y = full_train['Class'] # classifier

# functions for variance and correlation removal

def variance(data, threshold=(0.1)):

    sel = VarianceThreshold(threshold)  # removing features of low variance
    sel.fit_transform(data)
    
    return data[data.columns[sel.get_support(indices=True)]]

def findCorrelation(corr, cutoff=0.9, exact=None):
    
    def _findCorrelation_fast(corr, avg, cutoff):

        combsAboveCutoff = corr.where(lambda x: (np.tril(x)==0) & (x > cutoff)).stack().index

        rowsToCheck = combsAboveCutoff.get_level_values(0)
        colsToCheck = combsAboveCutoff.get_level_values(1)

        msk = avg[colsToCheck] > avg[rowsToCheck].values
        deletecol = pd.unique(np.r_[colsToCheck[msk], rowsToCheck[~msk]]).tolist()

        return deletecol


    def _findCorrelation_exact(corr, avg, cutoff):

        x = corr.loc[(*[avg.sort_values(ascending=False).index]*2,)]

        if (x.dtypes.values[:, None] == ['int64', 'int32', 'int16', 'int8']).any():
            x = x.astype(float)

        x.values[(*[np.arange(len(x))]*2,)] = np.nan

        deletecol = []
        for ix, i in enumerate(x.columns[:-1]):
            for j in x.columns[ix+1:]:
                if x.loc[i, j] > cutoff:
                    if x[i].mean() > x[j].mean():
                        deletecol.append(i)
                        x.loc[i] = x[i] = np.nan
                    else:
                        deletecol.append(j)
                        x.loc[j] = x[j] = np.nan
        return deletecol

    
    if not np.allclose(corr, corr.T) or any(corr.columns!=corr.index):
        raise ValueError("correlation matrix is not symmetric.")
        
    acorr = corr.abs()
    avg = acorr.mean()
        
    if exact or exact is None and corr.shape[1]<100:
        return _findCorrelation_exact(acorr, avg, cutoff)
    else:
        return _findCorrelation_fast(acorr, avg, cutoff)

# variance

train = full_train.drop(columns=['Structure','Class'])
train_var = variance(train) # removing features with low variance (threshold = 0.1)

# correlation

train_corr = train_var.corr()   # correlation matrix

hc = findCorrelation(train_corr, cutoff=0.9, exact=True)
train_var_corr = train_var.drop(columns=hc)     # removing highly correlated features (cutoff = 0.9)

# scaling

RobScaler = RobustScaler(unit_variance=True)    # fitting a robust scaler to the training data
RobScaler.fit(train_var_corr)

train_var_corr_scal = RobScaler.transform(train_var_corr) # scaling the training data
train_var_corr_scal = pd.DataFrame(train_var_corr_scal, columns=train_var_corr.columns)

str_clf = pd.concat([df_str, df_y], axis=1) # dataframe with smiles and class
str_clf = str_clf.reset_index(drop=True)

df2 = pd.concat([str_clf, train_var_corr_scal], axis=1) # dataframe with smiles, class and scaled descriptors

# downsampling (balances the dataset, 1:0)

nearMiss = NearMiss(sampling_strategy='majority', version=2, n_neighbors=5, n_jobs=-1) # https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.NearMiss.html

df2_x = df2.drop(columns=['Structure'])
df2_y = df2['Class']

df2_str = pd.DataFrame(df2['Structure'])

df2_x_res, df2_y_res = nearMiss.fit_resample(df2_x, df2_y) # resampling the training data using NearMiss

index = nearMiss.sample_indices_    # indices of the samples selected by NearMiss
df2_smiles = df2_str.iloc[index]    # smiles of the selected samples
df2_smiles = df2_smiles.reset_index(drop=True)

df_res = pd.concat([df2_smiles, df2_x_res], axis=1)     # resampled dataframe with smiles, class and scaled descriptors

# test set processing

df_test = full_test[df_res.columns] # test set with the same columns as the training set

test_smiles = df_test['Structure']
test_y = df_test['Class']

df_test_scal = RobScaler.transform(df_test.drop(columns=['Structure', 'Class']))    # scaling the test set using the same scaler as for the training set
df_test_scal = pd.DataFrame(df_test_scal, columns=df_test.drop(columns=['Structure', 'Class']).columns)

test_smi_class = pd.concat([test_smiles, test_y], axis=1)
test_smi_class = test_smi_class.reset_index(drop=True)

test_set = pd.concat([test_smi_class, df_test_scal], axis=1)    # final test set with smiles, class and scaled descriptors